# 02 — Build Tutor Reference Landmarks (+ ST-GCN embeddings)

For each lesson:
1. Pick a clean reference clip from WLASL.
2. Extract a 32-frame MediaPipe Holistic landmark sequence (1629-D / frame).
3. (If notebook 03 has exported `encoder.onnx`) also pool a 256-D embedding so `tutor_scorer.py` can blend DTW + embedding-cosine similarity.

Output: `models/reference_signs.json` with shape
```
{ lesson_id: { 'landmarks': [[...], ...], 'embedding': [...] } }
```
Backwards compatible: scorer also accepts the old plain-list format.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/dl_project'
!pip install -q mediapipe opencv-python-headless datasets onnxruntime huggingface_hub

In [ ]:
import os, json, cv2, numpy as np, mediapipe as mp
from datasets import load_from_disk

mp_hol = mp.solutions.holistic
holistic = mp_hol.Holistic(static_image_mode=False, model_complexity=1)

def landmarks_from_frame(rgb):
    r = holistic.process(rgb)
    pose = np.zeros((33, 4)); lh = np.zeros((21, 3)); rh = np.zeros((21, 3)); face = np.zeros((468, 3))
    if r.pose_landmarks:
        pose = np.array([[p.x, p.y, p.z, p.visibility] for p in r.pose_landmarks.landmark])
    if r.left_hand_landmarks:
        lh = np.array([[p.x, p.y, p.z] for p in r.left_hand_landmarks.landmark])
    if r.right_hand_landmarks:
        rh = np.array([[p.x, p.y, p.z] for p in r.right_hand_landmarks.landmark])
    if r.face_landmarks:
        face = np.array([[p.x, p.y, p.z] for p in r.face_landmarks.landmark])
    return np.concatenate([pose.flatten(), lh.flatten(), rh.flatten(), face.flatten()]).astype(np.float32)

def landmarks_from_video(path, n=32):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(total-1, 0), n).astype(int)
    out = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, f = cap.read()
        if not ok: out.append(np.zeros(1629, dtype=np.float32)); continue
        out.append(landmarks_from_frame(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
    cap.release()
    return np.stack(out)

In [ ]:
LESSONS = ['hello', 'thank-you', 'yes', 'no', 'please', 'sorry',
           'friend', 'family', 'eat', 'drink', 'help', 'school',
           'home', 'work', 'love', 'good', 'bad', 'happy', 'sad',
           'water', 'food', 'mother', 'father', 'sister', 'brother',
           'morning', 'night', 'today', 'tomorrow', 'yesterday']

train = load_from_disk(f'{BASE}/data/wlasl_top300_train')
lookup = {ex['gloss'].lower().replace(' ', '-'): ex for ex in train}

raw = {}
for lesson in LESSONS:
    ex = lookup.get(lesson)
    if ex is None:
        print('SKIP (no example):', lesson); continue
    path = ex['video']['path'] if isinstance(ex['video'], dict) else ex['video']
    seq = landmarks_from_video(path, n=32)
    raw[lesson] = seq
    print('built:', lesson, seq.shape)

In [ ]:
ENCODER = f'{BASE}/models/sthgcn_wlasl300/encoder.onnx'
embeddings = {}
if os.path.exists(ENCODER):
    import onnxruntime as ort
    sess = ort.InferenceSession(ENCODER, providers=['CPUExecutionProvider'])
    name = sess.get_inputs()[0].name
    for lesson, seq in raw.items():
        x = seq.astype(np.float32)[None, ...]
        (emb,) = sess.run(None, {name: x})
        embeddings[lesson] = emb.squeeze().tolist()
    print('embeddings ready for', len(embeddings), 'lessons')
else:
    print('encoder.onnx not found — skipping embedding pass (DTW-only scoring will be used).')

In [ ]:
blob = {lesson: {'landmarks': raw[lesson].tolist(),
                 **({'embedding': embeddings[lesson]} if lesson in embeddings else {})}
        for lesson in raw}
json.dump(blob, open(f'{BASE}/models/reference_signs.json', 'w'))
print('Saved', len(blob), 'references.')

from huggingface_hub import HfApi, create_repo
HF_ORG = 'uet-signlang'
REPO = f'{HF_ORG}/sign-tutor-refs'
create_repo(REPO, repo_type='model', private=True, exist_ok=True)
HfApi().upload_file(path_or_fileobj=f'{BASE}/models/reference_signs.json',
                    path_in_repo='reference_signs.json',
                    repo_id=REPO, repo_type='model')
print('Pushed to', REPO)